In [2]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import pandas as pd
import seaborn as sns
import sys
import os
from pathlib import Path

In [3]:
sys.path.insert(1, os.path.abspath(".."))

In [4]:
lf = pl.scan_parquet("../data/preprocessed_all_data.parquet")

In [5]:
# lf = df_all.lazy()
df_all = lf.collect()

In [65]:

# The "Anchor" for your Rolling Window
expr_culture_orders = (
    # Look for the ORDER or PROCEDURE
    (pl.col("event_full_name").str.to_lowercase().str.contains("order")) &
    (pl.col("event_full_name").str.to_lowercase().str.contains("culture")) &
    
    # Filter for the actual REQUEST (Procedure), not the Result
    (pl.col("event_full_name").str.to_lowercase().str.contains("procedure"))
)


expr_antibiotics_gold_standard = (
    (pl.col("Event_Grouper") == "Antibiotics") & 
    
    # 1. INCLUSION: Must be a systemic route
    (pl.col("event_full_name").str.to_lowercase().str.contains(r"\biv\b|intravenous|infusion|injection|push|piggy back"))&
    
    # 2. EXCLUSION: The "Not Sepsis" Filter
    ~(pl.col("event_full_name").str.to_lowercase().str.contains(
        r"dialysis|"        # Maintenance fluids
        r"heparin|"         # Line flushes
        r"lock solution|"   # Catheter cleaning
        r"chemo|"           # Cancer treatment
        r"rubicin|"         # Specific chemo agents (Doxorubicin, etc.)
        r"epoch|"           # Chemo cocktail
        r"intravitreal|"    # Eye injections (Local)
        r"intrapleural|"    # Lung cavity (Local/Mechanical)
        r"alteplase|"       # Clot busters
        r"activase"         # Clot busters
    ))
)

def detect_sepsis_suspicion(lf: pl.LazyFrame, expr_antibiotics_gold_standard: pl.Expr,
                            expr_culture_orders: pl.Expr):
    lf_abx = lf.filter(
        expr_antibiotics_gold_standard
    ).select(
        "PAT_ENC_CSN_ID",
        pl.col("Event_DateTime").alias("ABX_Time")
    ).sort("ABX_Time")

    lf_culture = lf.filter(
        expr_culture_orders 
    ).select(
        "PAT_ENC_CSN_ID",
        pl.col("Event_DateTime").alias("culture_Time")
    ).sort("culture_Time")

    lf_suspected_backward = lf_abx.join_asof(
        lf_culture,
        left_on="ABX_Time",
        right_on="culture_Time",
        by="PAT_ENC_CSN_ID",
        strategy="backward",
        tolerance="24h"
    ).filter(pl.col("culture_Time").is_not_null())

    lf_suspected_forward = lf_abx.join_asof(
        lf_culture,
        left_on="ABX_Time",
        right_on="culture_Time",
        by="PAT_ENC_CSN_ID",
        strategy="forward",
        tolerance="72h"
    ).filter(pl.col("culture_Time").is_not_null())

    # Combine matches and pick the EARLIEST of the two times
    df_suspected_infection = (
        pl.concat([lf_suspected_backward, lf_suspected_forward])
        .unique()
        .with_columns(
            # Sepsis Time Zero = Min(Abx Time, Culture Time)
            pl.min_horizontal(["ABX_Time", "culture_Time"]).alias("Suspected_Infection_Time")
        )
        # Get the FIRST episode per encounter
        .group_by("PAT_ENC_CSN_ID")
        .agg(pl.col("Suspected_Infection_Time").min())
        .collect()
    )
    return df_suspected_infection, lf_suspected_backward, lf_suspected_forward

df_suspected_infection, lf_suspected_backward, lf_suspected_forward = detect_sepsis_suspicion(
    # df_all.lazy(),
    lf,
    expr_antibiotics_gold_standard, expr_culture_orders
)
print(df_suspected_infection.shape)

(3595, 2)


sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


In [6]:
lf_suspected = pl.concat([lf_suspected_backward, lf_suspected_forward]).collect()

In [9]:
df_all.filter(
    (pl.col("Event_Grouper") == 'Antibiotics')&
    (pl.col("event_full_name")).str.to_lowercase().str.contains("povidone-iodine")
)['event_full_name'].unique().to_list()

['Order - Medication__POVIDONE-IODINE 7.5 % TOPICAL SOLUTION__Antibiotics',
 'Order - Medication__POVIDONE-IODINE 10 % TOPICAL SWAB__Antibiotics',
 'Medication - Administration__POVIDONE-IODINE 10 % TOPICAL SOLN__Antibiotics',
 'Medication - Administration__POVIDONE-IODINE 10 % OINTMENT__Antibiotics',
 'Medication - Administration__POVIDONE-IODINE 10 % TOPICAL SWAB__Antibiotics',
 'Order - Medication__POVIDONE-IODINE 10 % TOPICAL SOLN__Antibiotics',
 'Order - Medication__POVIDONE-IODINE 10 % OINTMENT__Antibiotics']

In [66]:
# enc_with_antibiotic = df_all.filter(expr_antibiotics_gold_standard)['PAT_ENC_CSN_ID'].unique().to_list()
enc_with_antibiotic = df_all.filter(expr_antibiotics_gold_standard)['PAT_ENC_CSN_ID'].unique().to_list()
enc_with_culture = df_all.filter(expr_culture_orders)['PAT_ENC_CSN_ID'].unique().to_list()
enc_with_antibiotic_culture = set(enc_with_antibiotic).intersection(set(enc_with_culture))

In [74]:
df_all.filter(pl.col("Event_Grouper") == 'Antibiotics')['event_full_name'].unique().to_list()

['Order - Medication__PENICILLIN G POT IN DEXTROSE 2,000,000 UNIT/50 ML IV PIGGY BACK__Antibiotics',
 'Order - Medication__AMOXICILLIN 500 MG CAP__Antibiotics',
 'Medication - Administration__PIPERACILLIN-TAZOBACTAM-DEXTROSE (ISO) 3.375 GRAM/50 ML IV PIGGY BACK__Antibiotics',
 'Order - Medication__CHLORHEXIDINE (HIBICLENS) 4 % TOPICAL LIQUID (OR)__Antibiotics',
 'Order - Medication__DAPTOMYCIN IV PUSH__Antibiotics',
 'Medication - Administration__MUPIROCIN 2 % TOPICAL CREAM__Antibiotics',
 'Order - Medication__ERYTHROMYCIN ETHYLSUCCINATE 400 MG/5 ML SUSP, RECON__Antibiotics',
 'Order - Medication__PERITONEAL DIALYSIS MIXTURE SOLUTION__Antibiotics',
 'Order - Medication__PIPERACILLIN-TAZOBACTAM 13.5 GRAM INTRAVENOUS SOLUTION__Antibiotics',
 'Medication - Administration__AZITHROMYCIN 250 MG TAB__Antibiotics',
 'Order - Medication__AMPHOTERICIN B 50 MG SOLUTION FOR INJECTION__Antibiotics',
 'Medication - Administration__BACITRACIN ZINC-POLYMYXIN B 500 UNIT-10,000 UNIT/GRAM OINTMENT__Antib

In [72]:
df_all.filter(pl.col("PAT_ENC_CSN_ID") == 727717399)

PAT_ENC_CSN_ID,PAT_MRN_ID,PAT_ID,Ethnicity,FirstRace,Sex,Coverage_Financial_Class_Grouper,Pt_Arrival,Admit_Time,DateAdmitKey,Patient_Age,Chief_Complaint_All,Before_Admit_YN,Admitted_From_ED,LOS_Days,Hours_in_ED,Event_DateTime,Type,EVENT_NAME,Event_Grouper,Result_Flag,MEAS_VALUE,Sepsis_Category,FLAG_DEATH_CLARITY,event_full_name
i64,i64,str,str,str,str,str,datetime[ns],datetime[ns],i64,f64,str,str,str,i64,i64,datetime[ns],str,str,str,str,str,str,i64,str
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""Yes""","""No""",10,null,2025-05-19 12:28:00,"""Diagnosis Audit - Add""","""Cholangiocarcinoma (*)""","""Other""",null,null,"""NPOA-3""",0,"""Diagnosis Audit - Add__Cholang…"
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""No""","""No""",10,null,2025-05-19 12:51:00,"""Order - Medication""","""ACETAMINOPHEN 500 MG TAB""","""Analgesic Non-Narcotics""",null,null,"""NPOA-3""",0,"""Order - Medication__ACETAMINOP…"
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""No""","""No""",10,null,2025-05-19 12:52:45,"""Flowsheet""","""RESPIRATIONS""","""Respirations""",null,"""16""","""NPOA-3""",0,"""Flowsheet__RESPIRATIONS__Respi…"
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""No""","""No""",10,null,2025-05-19 12:52:45,"""Flowsheet""","""BLOOD PRESSURE""","""Blood Pressure""",null,"""181/87""","""NPOA-3""",0,"""Flowsheet__BLOOD PRESSURE__Blo…"
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""No""","""No""",10,null,2025-05-19 12:52:45,"""Flowsheet""","""TEMPERATURE""","""Temperature""",null,"""97.9""","""NPOA-3""",0,"""Flowsheet__TEMPERATURE__Temper…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""No""","""No""",10,null,2025-05-29 11:15:18,"""Flowsheet""","""PULSE""","""Pulse""",null,"""55""","""NPOA-3""",0,"""Flowsheet__PULSE__Pulse"""
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""No""","""No""",10,null,2025-05-29 11:15:18,"""Flowsheet""","""TEMPERATURE""","""Temperature""",null,"""97.9""","""NPOA-3""",0,"""Flowsheet__TEMPERATURE__Temper…"
727717399,93320420,"""Z5778919""","""Non-Hispanic/Latino""","""Unavailable/Unknown""","""Male""","""Medicare""",2025-05-19 12:28:48,2025-05-19 12:43:48,20250519,70.206707,null,"""No""","""No""",10,null,2025-05-29 12:17:00,"""Flowsheet""","""UTSW UH R UNMEASURED URINE OCC…","""Urine Output""",null,"""1""","""NPOA-3""",0,"""Flowsheet__UTSW UH R UNMEASURE…"


In [10]:
set(npoa3_enc)-set(enc_with_antibiotic)

NameError: name 'npoa3_enc' is not defined

In [69]:

df_all.filter(pl.col("PAT_ENC_CSN_ID").is_in(
    set(npoa3_enc)-set(enc_with_antibiotic)
)).group_by("PAT_ENC_CSN_ID").agg(pl.len().alias('count'),  pl.col("Pt_Arrival").first(), pl.col("Event_DateTime").last(), pl.col("Patient_Age").last()).with_columns(
    (pl.col("Event_DateTime")-pl.col("Pt_Arrival")).dt.total_days()
)

PAT_ENC_CSN_ID,count,Pt_Arrival,Event_DateTime,Patient_Age
i64,u64,datetime[ns],i64,f64
727717399,1560,2025-05-19 12:28:48,12,70.206707
729299844,4071,2025-06-11 15:15:00,21,60.303901
722093996,3874,2025-02-25 05:55:00,55,80.043805
722181771,1526,2025-02-25 17:44:00,16,79.047227
722561454,998,null,null,38.696783
…,…,…,…,…
732141379,1903,2025-07-26 14:51:00,15,71.101984
728885185,969,2025-06-09 09:05:38,6,59.255304
728630643,1576,2025-06-03 03:17:58,11,72.238193


In [64]:
df_all.filter(pl.col("PAT_ENC_CSN_ID") == 723816642)

PAT_ENC_CSN_ID,PAT_MRN_ID,PAT_ID,Ethnicity,FirstRace,Sex,Coverage_Financial_Class_Grouper,Pt_Arrival,Admit_Time,DateAdmitKey,Patient_Age,Chief_Complaint_All,Before_Admit_YN,Admitted_From_ED,LOS_Days,Hours_in_ED,Event_DateTime,Type,EVENT_NAME,Event_Grouper,Result_Flag,MEAS_VALUE,Sepsis_Category,FLAG_DEATH_CLARITY,event_full_name
i64,i64,str,str,str,str,str,datetime[ns],datetime[ns],i64,f64,str,str,str,i64,i64,datetime[ns],str,str,str,str,str,str,i64,str
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""Yes""","""Yes""",38,8,2025-03-23 17:11:00,"""Flowsheet""","""CPM S24 R AS SC GLASGOW COMA S…","""Glasgow Coma Score""",null,"""15""","""NPOA-3""",0,"""Flowsheet__CPM S24 R AS SC GLA…"
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""Yes""","""Yes""",38,8,2025-03-23 17:12:42,"""Flowsheet""","""WEIGHT/SCALE""","""Weight""",null,"""3360""","""NPOA-3""",0,"""Flowsheet__WEIGHT/SCALE__Weigh…"
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""Yes""","""Yes""",38,8,2025-03-23 17:12:42,"""Flowsheet""","""BLOOD PRESSURE""","""Blood Pressure""",null,"""122/60""","""NPOA-3""",0,"""Flowsheet__BLOOD PRESSURE__Blo…"
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""Yes""","""Yes""",38,8,2025-03-23 17:12:42,"""Flowsheet""","""PULSE""","""Pulse""",null,"""81""","""NPOA-3""",0,"""Flowsheet__PULSE__Pulse"""
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""Yes""","""Yes""",38,8,2025-03-23 17:12:42,"""Flowsheet""","""TEMPERATURE""","""Temperature""",null,"""98.4""","""NPOA-3""",0,"""Flowsheet__TEMPERATURE__Temper…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""No""","""Yes""",38,8,2025-05-01 15:20:00,"""Flowsheet""","""BLOOD PRESSURE""","""Blood Pressure""",null,"""120/53""","""NPOA-3""",0,"""Flowsheet__BLOOD PRESSURE__Blo…"
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""No""","""Yes""",38,8,2025-05-01 15:20:00,"""Flowsheet""","""PULSE""","""Pulse""",null,"""88""","""NPOA-3""",0,"""Flowsheet__PULSE__Pulse"""
723816642,98781967,"""Z11241318""","""Non-Hispanic/Latino""","""White""","""Female""","""Medicare""",2025-03-23 17:10:00,2025-03-24 01:34:00,20250324,68.396988,"""ABDOMINAL PAIN""","""No""","""Yes""",38,8,2025-05-01 15:20:00,"""Flowsheet""","""RESPIRATIONS""","""Respirations""",null,"""18""","""NPOA-3""",0,"""Flowsheet__RESPIRATIONS__Respi…"


In [62]:
set(npoa3_enc)-set(enc_with_antibiotic_culture)

{723816642}

In [58]:
df_all.filter(pl.col("PAT_ENC_CSN_ID")==722093996).filter(pl.col("Event_Grouper")=='Antibiotics')['EVENT_NAME'].to_list()

['MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'MUPIROCIN 2 % OINTMENT',
 'PIPERACILLIN-TAZOBACTAM 3.375G IVPB (MINIBAG PLUS)',
 'LINEZOLID 600 MG/300 ML IV',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'LINEZOLID 600 MG/300 ML IV',
 'LINEZOLID 600 MG/300 ML IV',
 'MUPIROCIN 2 % OINTMENT',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'LINEZOLID 600 MG/300 ML IV',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PLUS)',
 'PIPERACILLIN-TAZOBACTAM 4.5G IVPB (MINIBAG PL

In [44]:
enc_with_antibiotic_culture = df_all.filter(expr_antibiotics_gold_standard& expr_culture_orders)['PAT_ENC_CSN_ID'].unique()

In [46]:
len(npoa3_enc)

313

In [45]:
LEN(set(npoa3_enc) - set(enc_with_antibiotic_culture)

{695960811,
 697718727,
 699476058,
 705630549,
 708553079,
 709516764,
 710566991,
 711219567,
 711291066,
 711634600,
 711743163,
 712736291,
 712990475,
 713249637,
 713494393,
 713754181,
 713809904,
 713896995,
 713986813,
 714258072,
 714258471,
 714349177,
 714356035,
 714398067,
 714534044,
 714658799,
 714670948,
 714679392,
 714738435,
 714800085,
 714871212,
 714872125,
 714901045,
 714936647,
 715080844,
 715131999,
 715178956,
 715206321,
 715261758,
 715278022,
 715675849,
 715683762,
 715936275,
 715961369,
 716057615,
 716112176,
 716155089,
 716170408,
 716208986,
 716230893,
 716254313,
 716378803,
 716386924,
 716426887,
 716513136,
 716522133,
 716603434,
 716629145,
 716745079,
 716791205,
 716791386,
 716802228,
 716826509,
 716828653,
 716907686,
 717006646,
 717036079,
 717138018,
 717187915,
 717264691,
 717438884,
 717452217,
 717576078,
 717740076,
 717767063,
 717856271,
 717937894,
 718016274,
 718021638,
 718022292,
 718092252,
 718200023,
 718334950,
 718

In [28]:
npoa3_enc = df_all.filter((pl.col("Sepsis_Category") == 'NPOA-3')['PAT_ENC_CSN_ID'].unique().to_list()

In [30]:
suspected_enc = df_suspected_infection['PAT_ENC_CSN_ID'].unique().to_list()

In [ ]:
pl

In [31]:
set(npoa3_enc)-set(suspected_enc)

{721526717,
 722093996,
 722181771,
 722523701,
 722561454,
 723816642,
 724267617,
 727717399,
 728630643,
 728885185,
 729299844,
 732141379,
 732504645}

In [63]:
df_all.filter(pl.col("PAT_ENC_CSN_ID").is_in(
    (set(npoa3_enc)-set(suspected_enc)).union({723816642})
)).group_by("PAT_ENC_CSN_ID").agg(pl.len().alias('count'),  pl.col("Pt_Arrival").first(), pl.col("Event_DateTime").last()).with_columns(
    (pl.col("Event_DateTime")-pl.col("Pt_Arrival")).dt.total_days()
)

PAT_ENC_CSN_ID,count,Pt_Arrival,Event_DateTime
i64,u64,datetime[ns],i64
728885185,969,2025-06-09 09:05:38,6
729299844,4071,2025-06-11 15:15:00,21
728630643,1576,2025-06-03 03:17:58,11
722561454,998,null,null
721526717,2274,2025-02-15 00:43:26,20
…,…,…,…
723816642,4141,2025-03-23 17:10:00,38
722093996,3874,2025-02-25 05:55:00,55
732504645,10450,2025-07-31 15:35:00,48


In [34]:
df_all.filter(pl.col("PAT_ENC_CSN_ID").is_in(
    set(npoa3_enc)-set(suspected_enc)
)).sort(by=['PAT_ENC_CSN_ID', 'Event_DateTime']).select(
    "PAT_ENC_CSN_ID", "Pt_Arrival", 'Admit_Time', 'Event_DateTime', 'EVENT_NAME'
)

PAT_ENC_CSN_ID,Pt_Arrival,Admit_Time,Event_DateTime,EVENT_NAME
i64,datetime[ns],datetime[ns],datetime[ns],str
721526717,2025-02-15 00:43:26,2025-02-15 03:33:36,2025-02-15 03:40:00,"""PULSE"""
721526717,2025-02-15 00:43:26,2025-02-15 03:33:36,2025-02-15 03:40:00,"""RESPIRATIONS"""
721526717,2025-02-15 00:43:26,2025-02-15 03:33:36,2025-02-15 03:40:00,"""BLOOD PRESSURE"""
721526717,2025-02-15 00:43:26,2025-02-15 03:33:36,2025-02-15 03:40:00,"""WEIGHT/SCALE"""
721526717,2025-02-15 00:43:26,2025-02-15 03:33:36,2025-02-15 03:40:00,"""MODEL R BMI"""
…,…,…,…,…
732504645,2025-07-31 15:35:00,2025-07-31 22:56:00,2025-09-17 17:58:00,"""RESPIRATIONS"""
732504645,2025-07-31 15:35:00,2025-07-31 22:56:00,2025-09-17 17:58:00,"""BLOOD PRESSURE"""
732504645,2025-07-31 15:35:00,2025-07-31 22:56:00,2025-09-17 18:00:00,"""CPM S24 R INV DEVICE (OXYGEN T…"


# SOFA score calculations

In [21]:
lf = lf.with_columns(
    pl.col("MEAS_VALUE")
    .str.extract(r"(-?\d+\.?\d*)", 1) # Regex to capture integer or float
    .cast(pl.Float64)
    .alias("val_numeric")
)

In [22]:
lf = lf.with_columns(
    pl.col("MEAS_VALUE")
    .str.extract(r"(-?\d+\.?\d*)", 1) # Regex to capture integer or float
    .cast(pl.Float64)
    .alias("val_numeric")
)

expr_bp = (
    pl.col("event_full_name").str.contains("BLOOD PRESSURE")
)

ALGO_bp_cond = (
    pl.col("event_full_name").str.contains("BLOOD PRESSURE")&
    pl.col("MEAS_VALUE").is_not_null()
)

lf = lf.join(
        lf.filter(
            ALGO_bp_cond
        ).with_columns(
            pl.col("MEAS_VALUE").str.split("/").list.get(0).cast(pl.Float64).alias("sys_bp"),
            pl.col("MEAS_VALUE").str.split("/").list.get(1).cast(pl.Float64).alias("dia_bp")
        ).select("PAT_ENC_CSN_ID", "Event_DateTime", "sys_bp", "dia_bp"),
        on=['PAT_ENC_CSN_ID', "Event_DateTime"],
        how='left'
).with_columns(
    MAP=((pl.col("sys_bp")+(2.0*pl.col("dia_bp")))/3.0).cast(pl.Float64)
)

In [19]:
lf.filter(
    pl.any_horizontal(
        [expr_plat, expr_bilirubin, expr_creatin, expr_bp]
    )
).collect()['event_full_name'].value_counts(sort=True)

event_full_name,count
str,u64
"""Flowsheet__BLOOD PRESSURE__Blo…",713804
"""Order Result - Lab__CREATININE…",80256
"""Order Result - Lab__PLATELETS_…",68705
"""Order Result - Lab__BILIRUBIN,…",33813
"""Order Result - Lab__PLT ESTIMA…",11252
"""Order Result - Lab__CREATININE…",294


In [14]:
lf.filter(
    expr_plat | expr_bilirubin | expr_creatin | expr_bp
).collect()['event_full_name'].unique()

event_full_name
str
"""Order Result - Lab__CREATININE…"
"""Flowsheet__BLOOD PRESSURE__Blo…"
"""Order Result - Lab__BILIRUBIN,…"
"""Order Result - Lab__CREATININE…"
"""Order Result - Lab__PLATELETS_…"
"""Order Result - Lab__PLT ESTIMA…"


In [11]:
# Platelet
expr_plat = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("platelet"))
)

expr_bilirubin = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("bili"))
)

expr_creatin = (
    (pl.col("event_full_name").str.to_lowercase().str.contains("creatin"))&
    (pl.col("event_full_name").str.to_lowercase().str.contains("order result"))
)

expr_bp = (
    pl.col("event_full_name").str.contains("BLOOD PRESSURE")
)

vasopressor_keywords = [
    "norepinephrine", "levophed", 
    "vasopressin", "vasostrict", 
    "phenylephrine", "neo-synephrine", 
    "dopamine", 
    "epinephrine", "adrenaline",  # <--- Added this (Critical for severe shock)
    "giapreza", "angiotensin"     # <--- Added this (Newer shock drug, rare but possible)
]

expr_vasopressors_final = (
    # 1. Match ANY of the drug names
    (pl.col("event_full_name").str.to_lowercase().str.contains_any(vasopressor_keywords)) &
    
    # 2. EXCLUDE "Push Doses" (Transient use)
    # This applies your specific Phenylephrine logic to ALL pressors just to be safe
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("push dose")) &
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("bolus")) &
    
    # 3. EXCLUDE Specific small vial sizes (implies push dose, based on your findings)
    ~(pl.col("event_full_name").str.to_lowercase().str.contains(r"\b10 mg\b")) & 
    ~(pl.col("event_full_name").str.to_lowercase().str.contains(r"\b1 mg\b")) &
    
    # 4. EXCLUDE Diagnosis/Audit text (Keep only Orders/Admin)
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("__other"))
)



In [ ]:
lf_labs_scored = lf_labs.with_columns(
    pl.when(pl.col("event_full_name").str.to_lowercase().str.contains("platelet"))
      .then(
          pl.when(pl.col("val_numeric") < 20).then(4)
            .when(pl.col("val_numeric") < 50).then(3)
            .when(pl.col("val_numeric") < 100).then(2)
            .when(pl.col("val_numeric") < 150).then(1)
            .otherwise(0)
      )
      .when(pl.col("event_full_name").str.to_lowercase().str.contains("bilirubin"))
      .then(
          pl.when(pl.col("val_numeric") >= 12.0).then(4)
            .when(pl.col("val_numeric") >= 6.0).then(3)
            .when(pl.col("val_numeric") >= 2.0).then(2)
            .when(pl.col("val_numeric") >= 1.2).then(1)
            .otherwise(0)
      )
      .when(pl.col("event_full_name").str.to_lowercase().str.contains("creatinine"))
      .then(
          pl.when(pl.col("val_numeric") >= 5.0).then(4)
            .when(pl.col("val_numeric") >= 3.5).then(3)
            .when(pl.col("val_numeric") >= 2.0).then(2)
            .when(pl.col("val_numeric") >= 1.2).then(1)
            .otherwise(0)
      )
      .alias("SOFA_Points")
      .cast(pl.Int32)
)

In [ ]:
expr_plat = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("platelet"))
)